In [1]:
import requests
import shap
import numpy as np
import shap.maskers

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma2:2b"

c:\Users\macie\LCNC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- funkcja wywołująca LLM ---
def query_llm(prompt):
    response = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "prompt": prompt,
        "stream": False
    })
    return response.json()["response"]


# --- metryka: jak bardzo odpowiedź jest "pozytywna" ---
# (tu: prosta heurystyka — możesz podmienić na embedding/logprob)
def score_output(output):
    positive_words = ["good", "great", "excellent", "positive"]
    score = sum(word in output.lower() for word in positive_words)
    return score


# --- funkcja dla SHAP ---
def model_fn(prompts):
    scores = []
    for p in prompts:
        out = query_llm(p)
        scores.append(score_output(out))
    return np.array(scores)

In [4]:
# --- prompt bazowy ---
prompt = """You are a helpful assistant.
Answer in a positive tone.
The product quality is average.
Give your opinion."""

# --- explainer ---
masker = shap.maskers.Text(" ")  # Define a masker for text data - splits on spaces
explainer = shap.Explainer(model_fn, masker=masker)

shap_values = explainer([prompt])


  0%|          | 0/210 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [04:44, 284.44s/it]              


In [5]:
shap.plots.text(shap_values[0])

In [8]:
from transformers import AutoTokenizer
prompt = """You are a helpful assistant.
Answer in a positive tone.
The product quality is average.
Give your opinion."""
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

masker = shap.maskers.Text(tokenizer)  # Use the tokenizer as the masker
explainer = shap.Explainer(model_fn, masker=masker)
shap_values = explainer([prompt])


PartitionExplainer explainer: 2it [15:33, 933.23s/it]              


In [ ]:
shap.plots.text(shap_values[0])

.values =
array([ 0.3125  ,  0.125   , -0.09375 ,  0.15625 , -0.0625  ,  0.125   ,
        0.1875  , -0.21875 ,  0.03125 ,  0.171875,  0.484375, -0.21875 ,
        0.      ,  0.03125 ,  0.09375 ,  0.140625, -0.078125,  0.0625  ,
        0.      , -0.109375, -0.203125,  0.125   ,  0.0625  , -0.125   ])

.base_values =
np.float64(0.0)

.data =
array(['', 'You ', 'are ', 'a ', 'helpful ', 'assistant', '.\n',
       'Answer ', 'in ', 'a ', 'positive ', 'tone', '.\n', 'The ',
       'product ', 'quality ', 'is ', 'average', '.\n', 'Give ', 'your ',
       'opinion', '.', ''], dtype=object)


In [3]:
import spacy
import shap

# Load SpaCy model
nlp = spacy.load("en_core_web_sm")

# Define a custom masker using SpaCy
def spacy_sentence_masker(text):
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]  # Tokenize into sentences
    return sentences

# Use the custom masker with SHAP
masker = shap.maskers.Text(spacy_sentence_masker)

# Create SHAP explainer
explainer = shap.Explainer(model_fn, masker=masker)

# Example text
prompt = """You are a helpful assistant.
Answer in a positive tone.
The product quality is average.
Give your opinion."""

# Generate SHAP values
shap_values = explainer([prompt])

# Visualize the explanation
shap.plots.text(shap_values[0])

TypeError: list indices must be integers or slices, not str